# Notebook 7: Citation Counts
We compare the total accumulated citations from publication to the last year with full citation data (2025). focusing on the difference between *sharing* and *non-sharing* articles, with a drill-down to the sharing subcategories (granularities).
<br><br>
Our analyses have two steps: First, we use a uni-variate approach (t-test or Mann-Whitney U test, depending on normality). Second, we fit a multivariable regression to test whether the findings hold when controlling for other known predictors of article impact (e.g., publication age, number of authors, open-access status, etc.).
<br><br>
_Note: the bottom section of this notebook stores a now-depricated analysis of citation residuals, which was used in the v1 manuscript but has been replaced by the multivariable regression above. It is kept here for reproducibility of the v1 results._

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.stats import compare_continuous

pio.renderers.default = "browser"

In [2]:
combined, FEATURES_DF = dataset.load_or_build()
CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)

CITATIONS_DF.columns

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


Index(['is_sharing_data', 'sharing_class', 'has_us_author', 'is_open_access',
       'has_preprint', 'venue_impact', 'log_weeks_since_pub',
       'log_number_of_authors', 'log_citations'],
      dtype='str')

### Univariate Comparison

In [3]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby("is_sharing_data")["log_citations"].describe().T

is_sharing_data,0,1
count,154.000000,80.000000
mean,2.656499,2.707260
std,0.889737,0.938678
min,0.000000,0.693147
25%,2.079442,2.197225
50%,2.639057,2.673554
75%,3.295837,3.433987
max,4.700480,4.762174


In [4]:
# check for normality
sw_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 1]["log_citations"])
print(f"Sharing Group is Normal:\t\t{'YES' if sw_share.pvalue > 0.05 else 'NO'} (W={sw_share.statistic :.3f}; p={sw_share.pvalue :.3f})")

sw_non_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 0]["log_citations"])
print(f"Non-Sharing Group is Normal:\t{'YES' if sw_non_share.pvalue > 0.05 else 'NO'} (W={sw_non_share.statistic :.3f}; p={sw_non_share.pvalue :.3f})")

if sw_share.pvalue > 0.05 and sw_non_share.pvalue > 0.05:
    print("Both groups are normally distributed, using t-test.")
else:
    print("At least one group is not normally distributed, using Mann-Whitney U test.")

Sharing Group is Normal:		YES (W=0.980; p=0.228)
Non-Sharing Group is Normal:	YES (W=0.988; p=0.223)
Both groups are normally distributed, using t-test.


In [24]:
# run univariate test
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF,
    share_feature="is_sharing_data",
    tested_feature="log_citations",
    alternative="greater",
    verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename("log_citations")
is_significant = citations_univariate_test_results.loc["p_val"] < 0.05

print(f"Univariate test is {'' if is_significant else 'NOT '}significant ({citations_univariate_test_results.loc['test']}={citations_univariate_test_results.loc['statistic']:.3f}, p={citations_univariate_test_results.loc['p_val']:.4f})")

citations_univariate_test_results

Univariate test is NOT significant (t=0.399, p=0.3451)


test                                                            t
alternative                                               greater
statistic                                                 0.39938
p_val                                                    0.345086
dof                                                     152.76833
effect_sizes    {'Cohen d': 0.05598459193658729, 'CLES': 0.515...
Name: log_citations, dtype: object

### Multivariable Regression
We compare *sharing* vs. *non-sharing* articles while controlling for other known predictors of citation impact, using the following regression model:

$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

In [26]:
citations_model = smf.ols(
    "log_citations ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint) + venue_impact + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    drop_cols=["sharing_class"]
).fit()
citations_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.166
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     6.416
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           6.76e-07
Time:                        11:47:31   Log-Likelihood:                -286.98
No. Observations:                 234   AIC:                             590.0
Df Residuals:                     226   BIC:                             617.6
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -4.6207      1.195     -3.866      0.000      -6.976      -2.265
C(is_sharing_data)[T.1]     0.0752      0.127      0.592      0.555      -0.175       0.325
C(has_us_author)[T.1]       0.1801      0.112      1.612      0.108      -0.040       0.400
C(is_open_access)[T.1]      0.3515      0.157      2.232      0.027       0.041       0.662
C(has_preprint)[T.1]        0.1657      0.144      1.151      0.251      -0.118       0.449
venue_impact                0.1099      0.045      2.443      0.015       0.021       0.199
log_weeks_since_pub         1.1266      0.199      5.675      0.000       0.735       1.518
log_number_of_authors       0.1066      0.132      0.810      0.419      -0.153       0.366
==============================================================================
Omnibus:                        4.298   Durbin-Watson:                   2.072
Prob(Omnibus):                  0.117   Jarque-Bera (JB):                4.040
Skew:                          -0.317   Prob(JB):                        0.133
Kurtosis:                       3.108   Cond. No.                         144.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [27]:
citations_model = smf.ols(
    "log_citations ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint) + C(venue_impact) + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    drop_cols=["sharing_class"]
).fit()
citations_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.315
Model:                            OLS   Adj. R-squared:                  0.222
Method:                 Least Squares   F-statistic:                     3.370
Date:                Fri, 09 Oct 2026   Prob (F-statistic):           2.98e-07
Time:                        11:47:50   Log-Likelihood:                -263.89
No. Observations:                 234   AIC:                             585.8
Df Residuals:                     205   BIC:                             686.0
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
=========================================================================================================
                                            coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                                -4.3336      1.221     -3.549      0.000      -6.741      -1.926
C(is_sharing_data)[T.1]                  -0.0217      0.140     -0.155      0.877      -0.297       0.254
C(has_us_author)[T.1]                     0.1121      0.113      0.993      0.322      -0.110       0.335
C(is_open_access)[T.1]                    0.3105      0.195      1.589      0.114      -0.075       0.696
C(has_preprint)[T.1]                      0.1215      0.147      0.827      0.409      -0.168       0.411
C(venue_impact)[T.0.6490855712161459]    -2.2381      0.820     -2.730      0.007      -3.854      -0.622
C(venue_impact)[T.0.9675925925925926]    -1.4083      0.602     -2.340      0.020      -2.595      -0.222
C(venue_impact)[T.1.4193548387096775]    -0.6230      0.265     -2.353      0.020      -1.145      -0.101
C(venue_impact)[T.1.5422993492407808]    -0.0998      0.324     -0.308      0.759      -0.740       0.540
C(venue_impact)[T.1.625615763546798]     -0.8543      0.337     -2.536      0.012      -1.518      -0.190
C(venue_impact)[T.1.660377358490566]     -0.5370      0.391     -1.375      0.171      -1.307       0.233
C(venue_impact)[T.1.7686116700201209]    -0.4664      0.188     -2.482      0.014      -0.837      -0.096
C(venue_impact)[T.1.8461538461538465]    -0.5702      0.825     -0.691      0.490      -2.197       1.056
C(venue_impact)[T.2.024390243902439]      0.0011      0.280      0.004      0.997      -0.551       0.553
C(venue_impact)[T.2.069356872635561]      1.1615      0.825      1.408      0.161      -0.465       2.788
C(venue_impact)[T.2.45985401459854]      -1.0885      0.828     -1.314      0.190      -2.721       0.544
C(venue_impact)[T.2.992258064516129]      0.3692      0.268      1.380      0.169      -0.158       0.897
C(venue_impact)[T.3.010889292196007]      0.0523      0.281      0.186      0.853      -0.502       0.607
C(venue_impact)[T.3.190113601473749]     -0.0717      0.257     -0.280      0.780      -0.577       0.434
C(venue_impact)[T.3.4690265486725664]    -0.0314      0.347     -0.091      0.928      -0.715       0.652
C(venue_impact)[T.3.52]                  -0.0180      0.254     -0.071      0.943      -0.518       0.482
C(venue_impact)[T.3.6112146629388007]    -0.3268      0.256     -1.279      0.202      -0.831       0.177
C(venue_impact)[T.3.930423782416193]      0.9234      0.832      1.109      0.269      -0.718       2.565
C(venue_impact)[T.4.135102533172497]     -0.5671      0.827     -0.685      0.494      -2.198       1.064
C(venue_impact)[T.4.987941414792678]      0.1236      0.291      0.425      0.671      -0.450       0.697
C(venue_impact)[T.6.015]                  1.4724      0.824      1.787      0.075      -0.152       3.097
C(venue_i

### Sharing-Class Comparisons
We repeat the same two-step analysis for the three sharing classes (FIXATION, TRIAL, PARTICIPANT) to determine whether there are differences in citations between the different levels of sharing granularity.

In [7]:
# remove level "NONE" from the dataset
CITATIONS_GRANULARITY_DF = CITATIONS_DF.loc[CITATIONS_DF["sharing_class"] != "NONE"]

# reset the categorical axis with PARTICIPANT as baseline
CITATIONS_GRANULARITY_DF["sharing_class"] = CITATIONS_GRANULARITY_DF["sharing_class"].cat.remove_unused_categories()

# compare the levels
CITATIONS_GRANULARITY_DF.groupby("sharing_class")["log_citations"].describe()

,count,mean,std,min,25%,50%,75%,max
sharing_class,,,,,,,,
PARTICIPANT,23.0,2.714876,0.995560,0.693147,2.191013,2.995732,3.312696,4.762174
TRIAL,26.0,2.472695,0.927505,0.693147,2.108887,2.302585,3.044522,4.127134
FIXATION,31.0,2.898341,0.889432,1.098612,2.302585,2.944439,3.637586,4.304065


In [8]:
citations_granularity_results, _ = compare_continuous(
    data=CITATIONS_GRANULARITY_DF, share_feature="sharing_class", tested_feature="log_citations", alternative="two-sided", verbose=False
)
citations_granularity_results = pd.Series(citations_granularity_results).rename("log_citations")
citations_granularity_results

test                                    ANOVA
statistic                            1.472223
p_val                                  0.2358
dof                                   (2, 77)
effect_sizes    {'eta2': 0.03683115523541124}
Name: log_citations, dtype: object

In [9]:
citations_granularity_model = smf.ols(
    "log_citations ~ C(sharing_class) + C(has_us_author) + C(is_open_access) + C(has_preprint) + venue_impact + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_GRANULARITY_DF,
    drop_cols=["is_sharing_data"]
).fit()
contrast_ftest = citations_granularity_model.f_test('C(sharing_class)[T.TRIAL] = C(sharing_class)[T.FIXATION] = 0')

print(f"Do 'TRIAL' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.TRIAL]"] :.4f}")
print(f"Do 'FIXATION' and 'PARTICIPANT' vary: {'YES' if citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] < 0.05 else 'NO'}:\tp_unc={citations_granularity_model.pvalues.loc["C(sharing_class)[T.FIXATION]"] :.4f}")
print(f"Do 'TRIAL' and 'FIXATION' vary: {'YES' if contrast_ftest.pvalue < 0.05 else 'NO'}:\tp_unc={contrast_ftest.pvalue :.4f}")

citations_granularity_model.summary()

Do 'TRIAL' and 'PARTICIPANT' vary: NO:	p_unc=0.5216
Do 'FIXATION' and 'PARTICIPANT' vary: NO:	p_unc=0.1862
Do 'TRIAL' and 'FIXATION' vary: NO:	p_unc=0.4120


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.290
Model:                            OLS   Adj. R-squared:                  0.210
Method:                 Least Squares   F-statistic:                     3.624
Date:                Fri, 09 Oct 2026   Prob (F-statistic):            0.00138
Time:                        11:39:30   Log-Likelihood:                -94.252
No. Observations:                  80   AIC:                             206.5
Df Residuals:                      71   BIC:                             227.9
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -5.7718      1.964     -2.939      0.004      -9.687      -1.856
C(sharing_class)[T.TRIAL]        0.1727      0.268      0.644      0.522      -0.362       0.707
C(sharing_class)[T.FIXATION]     0.3363      0.252      1.335      0.186      -0.166       0.839
C(has_us_author)[T.1]            0.4850      0.206      2.349      0.022       0.073       0.897
C(is_open_access)[T.1]           0.2122      0.367      0.578      0.565      -0.520       0.945
C(has_preprint)[T.1]            -0.2560      0.204     -1.256      0.213      -0.662       0.150
venue_impact                     0.1369      0.083      1.654      0.103      -0.028       0.302
log_weeks_since_pub              1.3287      0.334      3.979      0.000       0.663       1.994
log_number_of_authors            0.0922      0.248      0.372      0.711      -0.402       0.586
==============================================================================
Omnibus:                        0.278   Durbin-Watson:                   2.119
Prob(Omnibus):                  0.870   Jarque-Bera (JB):                0.453
Skew:                          -0.091   Prob(JB):                        0.797
Kurtosis:                       2.680   Cond. No.                         141.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

---

## Age-Adjusted Citation Residuals (SUPERSEDED)

**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group regressions at
the bottom.

It regresses log-citations on log-age alone, then compares the residuals between sharing and
non-sharing articles. The multivariable model above replaces it, for two reasons. Age is only one
of several predictors of citation count, so residualizing on age alone leaves venue, preprint
status and the rest uncontrolled. And the two-stage form (fit, then test the residuals) gives the
right point estimate with the wrong standard errors, because the second stage treats fitted
residuals as though they were observed data.

These cells deliberately use `Pub2UpdateTime` for article age rather than
`weeks_through_shared_year`. That is what v1 was computed from, and changing it here would stop
this section reproducing the published numbers, which is its only remaining purpose.

### Citation Counts over Time
#### (1) Quantify the Benefit of Data Sharing
We want to quantify the citation benefit of data sharing over time.<br>
To do so we first fit a log-log regression model to all data points, predicting citation counts from time since publication: $$\log(\text{Citations} +1) = \beta_0 + \beta_1 \log(\text{Weeks Since Publication}) + \epsilon$$(we add $1$ to the citation count to avoid $\log(0)$)<br><br>
Then, we compute the residuals (i.e., the difference between observed and predicted citation counts) for each article, and compare the residuals between sharing and non-sharing articles and across types of data shared.

In [10]:
# fit log-linear model to all data
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)  # add 1 to avoid log(0)
slope, intercept = np.polyfit(x_log, y_log, deg=1)

# compute residuals
ylog_pred = intercept + slope * x_log
residuals = (y_log - ylog_pred).rename("citation_residuals")
combined_with_residuals = pd.concat([combined, residuals], axis=1)

# extract residual by share/not-share
non_sharing_residuals = combined_with_residuals.loc[
    ~combined_with_residuals["is_sharing_data"], "citation_residuals"
]
sharing_residuals = combined_with_residuals.loc[
    combined_with_residuals["is_sharing_data"], "citation_residuals"
]

In [11]:
# check for normality of residuals
print("Are citation residuals normally distributed for non-sharing articles?")
residuals_non_sharing_shapiro = stats.shapiro(non_sharing_residuals)
print(f"{"YES" if residuals_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_non_sharing_shapiro.statistic:.3f}, p={residuals_non_sharing_shapiro.pvalue:.4f})\n")

print("Are citation residuals normally distributed for sharing articles?")
residuals_sharing_shapiro = stats.shapiro(sharing_residuals)
print(f"{"YES" if residuals_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_sharing_shapiro.statistic:.3f}, p={residuals_sharing_shapiro.pvalue:.4f})\n")

# run Levene's test for homogeneity of variance
levene_test = stats.levene(non_sharing_residuals, sharing_residuals)
is_equal_var = levene_test.pvalue > 0.05
print(f"Levene's test for homogeneity of variance: W={levene_test.statistic:.3f}, p={levene_test.pvalue:.4f}")
print(f"Sample have {'EQUAL' if is_equal_var else 'UNEQUAL'} variances.")

Are citation residuals normally distributed for non-sharing articles?
YES (W=0.987, p=0.1705)

Are citation residuals normally distributed for sharing articles?
YES (W=0.981, p=0.2692)

Levene's test for homogeneity of variance: W=0.125, p=0.7239
Sample have EQUAL variances.


In [12]:
# run two-sample t-test
residuals_ttest = pg.ttest(
    non_sharing_residuals, sharing_residuals, paired=False, alternative="less", correction=~is_equal_var,
)
print("t-test:")
display(residuals_ttest)

# run Mann-Whitney U-test
residuals_mwtest = pg.mwu(non_sharing_residuals, sharing_residuals, alternative="less")
print("MW U-test:")
display(residuals_mwtest)

# calculate the percentage difference in mean citations
non_share_residuals_mean = non_sharing_residuals.mean()
share_residuals_mean = sharing_residuals.mean()
citation_ratio = np.exp(share_residuals_mean - non_share_residuals_mean)
percentage_diff = (citation_ratio - 1) * 100
print(f"Data-Sharing has {percentage_diff:.1f}% citations compared with Non-Sharing.")

t-test:


,T,dof,alternative,p_val,CI95,cohen_d,power
T_test,-1.377062,232,less,0.08491,"[-inf, 0.03]",0.189782,0.392885


MW U-test:


,U_val,alternative,p_val,RBC,CLES
MWU,5493.0,less,0.087405,-0.108279,0.55414


Data-Sharing has 17.8% citations compared with Non-Sharing.


#### Is there a Difference Between Types of Data Shared?

In [13]:
share_group_residuals = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_residuals = (
        combined_with_residuals
        .loc[combined_with_residuals["data_sharing_class"] == share_group, "citation_residuals"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_residuals.append(group_residuals)
    shapiro_result = stats.shapiro(group_residuals)
    print(f"Are citation-residuals (in log-log space) normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

print("Statistical Tests:")
# run one-way anova
print("One-Way ANOVA")
sharegroup_residuals_anova = stats.f_oneway(*share_group_residuals)
display(sharegroup_residuals_anova)

# run Kruskal-Wallis
print("Kruskal-Wallis")
sharegroup_residuals_kruskal = stats.kruskal(*share_group_residuals)
display(sharegroup_residuals_kruskal)

Are citation-residuals (in log-log space) normally distributed for FIXATION sharing articles?
YES (W=0.960, p=0.2947)

Are citation-residuals (in log-log space) normally distributed for TRIAL sharing articles?
YES (W=0.980, p=0.8664)

Are citation-residuals (in log-log space) normally distributed for PARTICIPANT sharing articles?
YES (W=0.948, p=0.2611)

Statistical Tests:
One-Way ANOVA


F_onewayResult(statistic=np.float64(0.9323861725314082), pvalue=np.float64(0.398011242955337))

Kruskal-Wallis


KruskalResult(statistic=np.float64(2.0006032653648447), pvalue=np.float64(0.3677684934423655))

#### Visualizing Citation Residuals

In [14]:
column_titles = ["Citations for Time Since Publication", "Citation Residuals by Data Sharing"]
residuals_fig = make_subplots(
    rows=1, cols=2, shared_xaxes=False, shared_yaxes=False,
    column_titles=column_titles, column_widths=[0.7, 0.3],
    horizontal_spacing=0.075,
)

# single-article data
for is_share in sorted(combined_with_residuals["is_sharing_data"].unique()):
    is_sharing_subset = combined_with_residuals[combined_with_residuals["is_sharing_data"] == is_share]
    legendgroup = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[legendgroup.upper()]
    # left: scatter plot of citations over time
    for share_class in is_sharing_subset["data_sharing_class"].unique():
        label = share_class.title() if is_share else "Not Sharing"
        symbol = {"FIXATION": "circle", "TRIAL": "triangle-up", "PARTICIPANT": "square", "NONE": "x", }[share_class]
        class_subset = is_sharing_subset.loc[combined_with_residuals["data_sharing_class"] == share_class]
        residuals_fig.add_trace(
            row=1, col=1,
            trace=go.Scatter(
                name=label, legendgroup=label,
                x=class_subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1),
                y=class_subset["TotalCitations"],
                mode="markers",
                marker=dict(color=CLASS_COLORS[share_class], symbol=symbol, size=8, ),
            )
        )
    residuals_fig.update_xaxes(
        row=1, col=1, title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10), tickfont=AXIS_TICK_FONT,
    )
    residuals_fig.update_yaxes(
        row=1, col=1, title=dict(text="Total Citations", font=AXIS_TITLE_FONT, standoff=5), tickfont=AXIS_TICK_FONT,
    )

    # right: violin plot of residuals
    label = "Sharing" if is_share else "Not Sharing"
    residuals_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x0=0, y=is_sharing_subset["citation_residuals"],  # keeping it in log-space for visualization & statistics
            name=label, legendgroup=label, scalegroup=1,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.95, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
            showlegend=False,
        )
    )
    residuals_fig.update_xaxes(
        row=1, col=2, title=None, showticklabels=False,
    )
    residuals_fig.update_yaxes(
        row=1, col=2,
        title=dict(text="log-Residual Citations", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
    )
    # add annotations
    residuals_fig.add_annotation(
        row=1, col=2,
        x=0.2 if is_share else -0.2,
        xanchor="left" if is_share else "right", xref="x",
        y=1.75, yanchor="top", yref="y2",
        text=f"{label.replace(" ", "<br>")}",
        font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )
    residuals_fig.add_annotation(   # p value annotation
        row=1, col=2,
        x=0.125, xanchor="left", xref="x",
        y=-2.5, yanchor="middle", yref="y2",
        text=f"<i>p={residuals_ttest["p_val"].iloc[0] :.3f}</i>",
        font=AXIS_TITLE_FONT,
        showarrow=False,
    )

# add log-log trendline for citations over time
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1  # invert log transform
x_orig = np.exp(x_logspace)
residuals_fig.add_trace(
    row=1, col=1,
    trace=go.Scatter(
        name="Expected<br>Citations",
        x=x_orig, y=y_pred,
        mode="lines",
        line=dict(color='black', width=2.5, dash='dash'),
    )
)

# update layout and show
for ann in residuals_fig.layout.annotations:
    if ann.text not in column_titles:
        continue
    x = -0.03 if ann.text == column_titles[0] else 0.67
    ann.update(dict(
        font=AXIS_TITLE_FONT,
        x=x, xanchor="left", xref="paper",
        y=1.1, yanchor="top", yref="paper",
))
residuals_fig.update_xaxes(zeroline=False,)
residuals_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
residuals_fig.update_xaxes(
    row=1, col=1, range=[140, 505]
)
residuals_fig.update_yaxes(
    row=1, col=1, range=[-5, 119]
)
residuals_fig.update_layout(
    width=1000, height=400,
    title=dict(
        text="<b>Estimating Number of Citations by Weeks Past Publication</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.99, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(
        visible=True,
        orientation="v",
        entrywidth=30,
        bgcolor='rgba(255,255,255,0.2)',
        borderwidth=2, bordercolor='rgba(0,0,0,1)',
        x=0.01, xanchor="left", xref="paper",
        y=1.0, yanchor="top", yref="paper",
        font=LEGEND_FONT, font_size=10,
    ),
    margin=dict(t=60, b=5, l=50, r=10, pad=0),
    template="plotly_white"
)
residuals_fig.show()

In [15]:
# flip to True to export this figure into `output/`
if False:
    save_figure(residuals_fig, "citation_residuals_by_data_sharing.png", width=1000, height=400)

#### (2) Regress each Sharing Group Separately
exploratory analysis: fit separate log-log regression models for each sharing group

In [16]:
citation_counts_fig = go.Figure()
for group in combined["data_sharing_class"].unique():
    subset = combined[combined["data_sharing_class"] == group]
    color = CLASS_COLORS[group]
    symbol = 'circle' if group != "NONE" else 'x'
    citation_counts_fig.add_trace(go.Scatter(
        name=group, legendgroup=group,
        x=subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1), y=subset["TotalCitations"],
        mode="markers", marker=dict(color=color, size=8, symbol=symbol),
    ))
    # fit log-log trendline per group
    x_log = np.log(subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
    y_log = np.log(subset['TotalCitations'] + 1)  # add 1 to avoid log(0)
    slope, intercept = np.polyfit(x_log, y_log, deg=1)
    x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
    y_logpred = intercept + slope * x_logspace
    y_pred = np.exp(y_logpred) - 1  # invert log transform
    x_orig = np.exp(x_logspace)
    citation_counts_fig.add_trace(go.Scatter(
        name=f"Trendline ({group})",
        x=x_orig, y=y_pred, mode="lines", line=dict(color=color, width=2, dash='dot'),
        showlegend=False,
    ))
    print("Group: {0: <12}".format(group) + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

# add general log-log trendline
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)
slope, intercept = np.polyfit(x_log, y_log, deg=1)
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1
x_orig = np.exp(x_logspace)
citation_counts_fig.add_trace(go.Scatter(
    name="Trendline (all data)",
    x=x_orig, y=y_pred, mode="lines", line=dict(color='black', width=2, dash='dash'),
))
print("{0: <19}".format("All Data:") + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

citation_counts_fig.update_layout(
    width=800, height=300,
    title=dict(
        text="Citation Counts over Time Since Publication", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
    ),
    xaxis=dict(
        title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Total Citation Counts", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=True,
    ),
    legend=dict(
        orientation="h", bgcolor='rgba(0,0,0,0)',
        x=0.5, xanchor="center", y=0.96, yanchor="bottom",
    ),
    margin=dict(t=50, b=10, l=10, r=10, pad=0),
)
citation_counts_fig

Group: FIXATION    Slope: 1.431,	Intercept: -5.36
Group: NONE        Slope: 0.810,	Intercept: -2.03
Group: TRIAL       Slope: 0.912,	Intercept: -2.64
Group: PARTICIPANT Slope: 1.425,	Intercept: -5.45
All Data:          Slope: 0.953,	Intercept: -2.82
